In [0]:
%sql
CREATE OR REPLACE TABLE workspace.open_meteo.silver_hourly_forecast_enriched
COMMENT 'Hourly forecast per Open-Meteo grid point with weather-code lookup, flags, lead time and rolling columns. Grain: latitude x longitude x forecast_time.'
AS
WITH base AS (
  SELECT
    f.*,
    d.description  AS weather_description,
    d.category     AS weather_category,
-- difference columns
    f.apparent_temperature - f.temperature_2m          AS feels_like_delta,
-- flags (thresholds are my picks; tune them)
    f.precipitation > 0                                AS is_precipitating,
    f.precipitation >= 7.6                             AS is_heavy_rain,   -- mm/h
    f.temperature_2m <= 0                              AS is_freezing,
    f.apparent_temperature >= 40                       AS is_extreme_heat,
    f.wind_speed_10m >= 50                             AS is_high_wind,    -- km/h
-- time parts (UTC unless you fetched with timezone=auto)
    DATE(f.forecast_time)                              AS forecast_date,
    HOUR(f.forecast_time)                              AS forecast_hour,
    dayofweek(f.forecast_time) IN (1, 7)               AS is_weekend,
-- lead time: how far ahead of the fetch this hour is
    (unix_timestamp(f.forecast_time) - unix_timestamp(f.fetched_at)) / 3600.0
                                                       AS lead_hours
  FROM workspace.open_meteo.silver_hourly_forecast f
  LEFT JOIN workspace.open_meteo.dim_weather_code d USING (weather_code)
)
SELECT
  *,
  -- windowed columns, per grid point in time order
  SUM(precipitation) OVER w3                           AS precip_next_3h,
  SUM(precipitation) OVER w24                          AS precip_next_24h,
  temperature_2m - LAG(temperature_2m) OVER w          AS temp_change_1h,
  MAX(temperature_2m) OVER dy                          AS day_max_temp,
  MIN(temperature_2m) OVER dy                          AS day_min_temp
FROM base
WINDOW
  w   AS (PARTITION BY latitude, longitude ORDER BY forecast_time),
  w3  AS (PARTITION BY latitude, longitude ORDER BY forecast_time ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING),
  w24 AS (PARTITION BY latitude, longitude ORDER BY forecast_time ROWS BETWEEN CURRENT ROW AND 23 FOLLOWING),
  dy  AS (PARTITION BY latitude, longitude, forecast_date);